# CSV 파일 처리

## 환경 설정

In [1]:
# 필요한 모듈 / 함수 / 클래스 임포트
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import CSVLoader
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

C:\Users\playdata2\AppData\Local\Temp\ipykernel_12444\3921460080.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


In [2]:
# .env file -> API Key 가져오기
env_file_path='.env'
output = load_dotenv(env_file_path)
print(output)

True


## 데이터 로드 및 변환

In [4]:
# 파일 경로 설정
csv_path='ChatBotData_sample.csv'

# CSV 파일 로드 및 변환 도구 생성 -> CSVLoader 클래스 호출
loader = CSVLoader(
    file_path=csv_path,
    encoding='utf-8',
    # 본문(page_content)에 포함할 컬럼 지정
    content_columns=["Q","A"],
    # 메타데이터로 뺄 컬럼 지정, label 컬럼은 메타데이터로 설정
    metadata_columns=['label'] 
)

# 전용 함수 load() 함수 호출 -> '행' 단위 -> Document 생성
docs = loader.load()

print(f"로드된 총 문서 수: {len(docs)}개")
print("-" * 60)
print(f"[첫 번째 Document 본문(page_content)]\n{docs[0].page_content}")
print("-" * 60)
print(f"[첫 번째 Document 메타데이터(metadata)]\n{docs[0].metadata}")

로드된 총 문서 수: 30개
------------------------------------------------------------
[첫 번째 Document 본문(page_content)]
Q: 12시 땡!
A: 하루가 또 가네요.
------------------------------------------------------------
[첫 번째 Document 메타데이터(metadata)]
{'source': 'ChatBotData_sample.csv', 'row': 0, 'label': '0'}


## 텍스트 임베딩 모델 생성

In [5]:
# 모델 이름 설정
model_name = "text-embedding-3-small"

# OpenAI 임베딩 모델 생성
model = OpenAIEmbeddings(
    model=model_name
)

## 백터 저장소 및 검색 도구 생성

In [6]:
# 백터 저장소 경로 설정
chroma_path = 'chroma_db_csv'

# 기존 DB가 존재하면 로드, 없으면 새로 생성
if os.path.exists(chroma_path) and os.listdir(chroma_path):
    vectorstore = Chroma(
        persist_directory=chroma_path,
        embedding_function=model
    )
    print("기존 벡터 저장소를 성공적으로 불러왔습니다.")
else:
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=model,
        persist_directory=chroma_path
    )
    print("신규 벡터 저장소를 생성하고 저장했습니다.")

신규 벡터 저장소를 생성하고 저장했습니다.


In [7]:
# 검색 도구 생성
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})

In [8]:
# 검색 도구 성능 테스트
user_query="가족 여행 갈거야. 기대가 많이 되"
output = retriever.invoke(user_query)
print(f'검색 결과: \n{output}')

검색 결과: 
[Document(id='d84e328f-5539-412f-bc12-3b584137c8f0', metadata={'label': '0', 'row': 23, 'source': 'ChatBotData_sample.csv'}, page_content='Q: 가족 여행 가기로 했어\nA: 온 가족이 모두 마음에 드는 곳으로 가보세요.'), Document(id='07cb2e92-a781-4b27-a443-6ab6c76e2e39', metadata={'row': 28, 'label': '0', 'source': 'ChatBotData_sample.csv'}, page_content='Q: 가족끼리 여행간다.\nA: 더 가까워질 기회가 되겠네요.')]
